In [47]:
# Importación de Librerías Esenciales

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge, Lasso, RidgeCV, LassoCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import lasso_path


<div class="alert alert-block alert-info">
    <h1>Taller Práctico: Regresión Lineal, Ridge y Lasso</h1>
    <p><strong>Objetivos de la sesión:</strong></p>
    <ul>
        <li>Aplicar los conceptos de regresión lineal en un problema práctico.</li>
        <li>Implementar y comparar modelos de Regresión Lineal, Ridge y Lasso.</li>
        <li>Utilizar la validación cruzada para encontrar el hiperparámetro de regularización óptimo.</li>
        <li>Evaluar el rendimiento de los modelos utilizando métricas como MSE, MAE y R².</li>
    </ul>
    <p><strong>Dataset:</strong> Utilizaremos el dataset "Ames Housing", que contiene información sobre la venta de casas en Ames, Iowa. Nuestro objetivo será predecir el precio de venta (<code>SalePrice</code>) de las casas.</p>
</div>

### 1. Carga y Exploración de Datos (EDA)

<p>El primer paso en cualquier proyecto de Machine Learning es entender nuestros datos. Cargaremos el dataset, veremos su estructura y realizaremos algunas visualizaciones iniciales.</p>

In [2]:
# Cargar el dataset Ames Housing desde OpenML
housing = fetch_openml(name="house_prices", as_frame=True)
df = housing.frame

# Seleccionamos un subconjunto de características numéricas para simplificar el taller
# y la variable objetivo 'SalePrice'
numeric_features = ['GrLivArea', 'OverallQual', 'YearBuilt', 'TotalBsmtSF', 'FullBath', 'GarageCars', 'YrSold', 'LotArea']
target_variable = 'SalePrice'

# Creamos un DataFrame más pequeño y manejamos valores faltantes de forma simple
df_subset = df[numeric_features + [target_variable]].dropna()

print(f"\nDimensiones del subconjunto de datos: {df_subset.shape}")
print("\nDescripción estadística del subconjunto:")
print(df_subset.describe())


Dimensiones del subconjunto de datos: (1460, 9)

Descripción estadística del subconjunto:
         GrLivArea  OverallQual    YearBuilt  TotalBsmtSF     FullBath  \
count  1460.000000  1460.000000  1460.000000  1460.000000  1460.000000   
mean   1515.463699     6.099315  1971.267808  1057.429452     1.565068   
std     525.480383     1.382997    30.202904   438.705324     0.550916   
min     334.000000     1.000000  1872.000000     0.000000     0.000000   
25%    1129.500000     5.000000  1954.000000   795.750000     1.000000   
50%    1464.000000     6.000000  1973.000000   991.500000     2.000000   
75%    1776.750000     7.000000  2000.000000  1298.250000     2.000000   
max    5642.000000    10.000000  2010.000000  6110.000000     3.000000   

        GarageCars       YrSold        LotArea      SalePrice  
count  1460.000000  1460.000000    1460.000000    1460.000000  
mean      1.767123  2007.815753   10516.828082  180921.195890  
std       0.747315     1.328095    9981.264932   7

<p>Ahora, visualicemos la distribución de nuestra variable objetivo y la relación entre el área de la vivienda y su precio. La distribución de precios está sesgada, por lo que una transformación logarítmica será útil.</p>

In [3]:
# Histograma del Precio de Venta (SalePrice)
fig_hist = px.histogram(df_subset, x='SalePrice', nbins=50, title='Distribución del Precio de Venta de las Casas (Original)')
fig_hist.update_layout(xaxis_title='Precio de Venta (USD)', yaxis_title='Cantidad de casas')
fig_hist.show()

# Histograma del log(Precio de Venta)
fig_hist_log = px.histogram(x=np.log1p(df_subset['SalePrice']), nbins=50, title='Distribución del log(Precio de Venta)')
fig_hist_log.update_layout(xaxis_title='log(1 + Precio de Venta)', yaxis_title='Cantidad de casas')
fig_hist_log.show()

### 2. Preparación de Datos

<p>Ahora aplicaremos los cambios necesarios: división de datos, transformación logarítmica del objetivo y escalado de características.</p>

In [4]:
# 1. Definir características (X) y objetivo (y)
X = df_subset[numeric_features]
y_raw = df_subset[target_variable] # Guardamos el 'y' original para evaluación final

# !! CAMBIO CLAVE !!
# Aplicamos la transformación logarítmica a la variable objetivo 'y'
# Esto estabiliza la varianza y normaliza la escala del problema.
y = np.log1p(y_raw)

# 2. Dividir en conjuntos de entrenamiento y prueba (80% entrenamiento, 20% prueba)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Tamaño del set de entrenamiento: {X_train.shape}")
print(f"Tamaño del set de prueba: {X_test.shape}")

# 3. Escalar las características numéricas
scaler = StandardScaler()

# Ajustamos el escalador SOLO con los datos de entrenamiento para evitar fuga de datos (data leakage)
X_train_scaled = scaler.fit_transform(X_train)
# Aplicamos la misma transformación a los datos de prueba
X_test_scaled = scaler.transform(X_test)

# Convertimos los arrays de numpy de vuelta a DataFrames de pandas para mejor legibilidad
X_train_scaled = pd.DataFrame(X_train_scaled, columns=numeric_features)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=numeric_features)

Tamaño del set de entrenamiento: (1168, 8)
Tamaño del set de prueba: (292, 8)


### 3. Regresión Lineal (Mínimos Cuadrados Ordinarios)

<p>Entrenamos nuestro modelo base con los datos transformados. El MSE será un número pequeño y manejable.</p>

In [5]:
# Crear y entrenar el modelo de Regresión Lineal
ols_model = LinearRegression()
ols_model.fit(X_train_scaled, y_train)

# Realizar predicciones en el conjunto de prueba (las predicciones estarán en escala logarítmica)
y_pred_ols = ols_model.predict(X_test_scaled)

# Evaluar el modelo
mse_ols = mean_squared_error(y_test, y_pred_ols)
r2_ols = r2_score(y_test, y_pred_ols)

print("--- Evaluación del Modelo de Regresión Lineal (OLS) ---")
print(f"Error Cuadrático Medio (MSE en escala log): {mse_ols:.4f}")
print(f"Coeficiente de Determinación (R²): {r2_ols:.4f}")

# Ver los coeficientes del modelo
coef_df = pd.DataFrame(ols_model.coef_, index=numeric_features, columns=['Coeficiente_OLS'])
print("\nCoeficientes del modelo OLS:")
print(coef_df)

--- Evaluación del Modelo de Regresión Lineal (OLS) ---
Error Cuadrático Medio (MSE en escala log): 0.0297
Coeficiente de Determinación (R²): 0.8410

Coeficientes del modelo OLS:
             Coeficiente_OLS
GrLivArea           0.118066
OverallQual         0.143974
YearBuilt           0.076205
TotalBsmtSF         0.036486
FullBath           -0.005818
GarageCars          0.069123
YrSold              0.000511
LotArea             0.034061


### 4. Regresión con Regularización: Ridge (L2) y Lasso (L1)

<p>Con el objetivo ya transformado, ahora `alpha` tendrá el efecto esperado. Usaremos un valor pequeño para ver el contraste.</p>

In [6]:
# --- Ridge Regression (L2) ---
ridge_model = Ridge(alpha=100.) # Alpha grande
ridge_model.fit(X_train_scaled, y_train)
y_pred_ridge = ridge_model.predict(X_test_scaled)

mse_ridge = mean_squared_error(y_test, y_pred_ridge)
r2_ridge = r2_score(y_test, y_pred_ridge)

print("--- Evaluación del Modelo Ridge (alpha=100.) ---")
print(f"Error Cuadrático Medio (MSE en escala log): {mse_ridge:.4f}")
print(f"Coeficiente de Determinación (R²): {r2_ridge:.4f}")

# --- Lasso Regression (L1) ---
lasso_model = Lasso(alpha=0.1) # Alpha pequeño
lasso_model.fit(X_train_scaled, y_train)
y_pred_lasso = lasso_model.predict(X_test_scaled)

mse_lasso = mean_squared_error(y_test, y_pred_lasso)
r2_lasso = r2_score(y_test, y_pred_lasso)

print("\n--- Evaluación del Modelo Lasso (alpha=0.1) ---")
print(f"Error Cuadrático Medio (MSE en escala log): {mse_lasso:.4f}")
print(f"Coeficiente de Determinación (R²): {r2_lasso:.4f}")

# Comparar coeficientes
coef_df['Coef_Ridge_alpha100'] = ridge_model.coef_
coef_df['Coef_Lasso_alpha0.1'] = lasso_model.coef_
print("\n--- Comparación de Coeficientes ---")
print(coef_df.round(4))

--- Evaluación del Modelo Ridge (alpha=100.) ---
Error Cuadrático Medio (MSE en escala log): 0.0304
Coeficiente de Determinación (R²): 0.8373

--- Evaluación del Modelo Lasso (alpha=0.1) ---
Error Cuadrático Medio (MSE en escala log): 0.0579
Coeficiente de Determinación (R²): 0.6895

--- Comparación de Coeficientes ---
             Coeficiente_OLS  Coef_Ridge_alpha100  Coef_Lasso_alpha0.1
GrLivArea             0.1181               0.1051               0.0567
OverallQual           0.1440               0.1324               0.1479
YearBuilt             0.0762               0.0692               0.0050
TotalBsmtSF           0.0365               0.0430               0.0060
FullBath             -0.0058               0.0086               0.0000
GarageCars            0.0691               0.0702               0.0486
YrSold                0.0005              -0.0001              -0.0000
LotArea               0.0341               0.0325               0.0000


<p> Fijate cómo los coeficientes de Ridge son sistemáticamente penalizados (encogidos) respecto de los de OLS. Y más importante, mirá los coeficientes de Lasso: con un `alpha` de solo 0.1, redujo el coeficiente de `FullBath` casi a cero, demostrando su poder de selección de características.</p>

### 5. Selección de Hiperparámetros con Validación Cruzada

<p>Finalmente, usamos `RidgeCV` y `LassoCV` para encontrar el `alpha` óptimo de manera automática.</p>

In [7]:
# Definir un rango de alphas para probar
alphas = np.logspace(-4, 2, 100)

# --- RidgeCV ---
ridge_cv = RidgeCV(alphas=alphas, store_cv_values=True)
ridge_cv.fit(X_train_scaled, y_train)

print("--- Búsqueda del mejor Alpha para Ridge ---")
print(f"Mejor alpha encontrado para Ridge: {ridge_cv.alpha_:.4f}")

# Evaluar el modelo Ridge final con el mejor alpha
y_pred_ridge_cv = ridge_cv.predict(X_test_scaled)
mse_ridge_cv = mean_squared_error(y_test, y_pred_ridge_cv)
r2_ridge_cv = r2_score(y_test, y_pred_ridge_cv)
print(f"MSE de Ridge con CV: {mse_ridge_cv:.4f}")
print(f"R² de Ridge con CV: {r2_ridge_cv:.4f}")

# --- LassoCV ---
lasso_cv = LassoCV(alphas=alphas, cv=5, random_state=42)
lasso_cv.fit(X_train_scaled, y_train)

print("\n--- Búsqueda del mejor Alpha para Lasso ---")
print(f"Mejor alpha encontrado para Lasso: {lasso_cv.alpha_:.4f}")

# Evaluar el modelo Lasso final
y_pred_lasso_cv = lasso_cv.predict(X_test_scaled)
mse_lasso_cv = mean_squared_error(y_test, y_pred_lasso_cv)
r2_lasso_cv = r2_score(y_test, y_pred_lasso_cv)
print(f"MSE de Lasso con CV: {mse_lasso_cv:.4f}")
print(f"R² de Lasso con CV: {r2_lasso_cv:.4f}")

# Añadir los coeficientes finales al DataFrame
coef_df['Coef_Ridge_CV'] = ridge_cv.coef_
coef_df['Coef_Lasso_CV'] = lasso_cv.coef_
print("\n--- Comparación Final de Coeficientes ---")
print(coef_df.round(4))

--- Búsqueda del mejor Alpha para Ridge ---
Mejor alpha encontrado para Ridge: 86.9749
MSE de Ridge con CV: 0.0302
R² de Ridge con CV: 0.8379

--- Búsqueda del mejor Alpha para Lasso ---
Mejor alpha encontrado para Lasso: 0.0043
MSE de Lasso con CV: 0.0301
R² de Lasso con CV: 0.8387

--- Comparación Final de Coeficientes ---
             Coeficiente_OLS  Coef_Ridge_alpha100  Coef_Lasso_alpha0.1  \
GrLivArea             0.1181               0.1051               0.0567   
OverallQual           0.1440               0.1324               0.1479   
YearBuilt             0.0762               0.0692               0.0050   
TotalBsmtSF           0.0365               0.0430               0.0060   
FullBath             -0.0058               0.0086               0.0000   
GarageCars            0.0691               0.0702               0.0486   
YrSold                0.0005              -0.0001              -0.0000   
LotArea               0.0341               0.0325               0.0000   

      

/Users/rodrigoecheconea/Projects/Master AI/aprendizaje_automatico/tp2/.venv/lib/python3.9/site-packages/sklearn/linear_model/_ridge.py:2385: FutureWarning: 'store_cv_values' is deprecated in version 1.5 and will be removed in 1.7. Use 'store_cv_results' instead.
  warnings.warn(


### 6. Desafíos para Experimentar

<p>Ahora es tu turno de experimentar. Respondé a las siguientes preguntas modificando el código anterior:</p>
<ol>
    <li><strong>Añadir más características:</strong> Elegí otras 2 o 3 features numéricos del `df` original, añadilos a la lista `numeric_features` y volvé a ejecutar todo el notebook. ¿Mejora el R² del modelo final?</li>
    <li><strong>Interpretación de coeficientes:</strong> Observá los coeficientes finales del modelo `LassoCV`. ¿Qué característica parece ser la más importante según este modelo? ¿Cuáles ha descartado (coeficiente cercano a cero)?</li>
    <li><strong>Evaluación en escala original:</strong> Las predicciones (`y_pred_lasso_cv`) están en escala logarítmica. Usá `np.expm1()` para convertirlas de nuevo a dólares. Luego, calculá el Error Absoluto Medio (MAE) entre las predicciones revertidas y el `y_test` original (que deberías haber guardado como `y_raw`). ¿Cuál es el error promedio en dólares de tu mejor modelo?</li>
</ol>

1) Sumamos LotArea y YrSold porque consideramos que el tamaño del lote puede agregar informacion y porque el año en que se vendio es fundamental para entender el precio de venta.
Luego de revisar los resultados concluimos que no mejoraron el R2, mas bien lo empeoraron de forma marginal.
2) La caracteritica mas importante es sin dudas el OverallQual, seguido por el GrLivArea. Los descartados por LassoCV fueron FullBath y el YrSold agregado por nosotros.
3) El MAE, usando LassoCV, es $20812

In [9]:
y_pred_reverted = np.expm1(y_pred_lasso_cv)
y_test_reverted = np.expm1(y_test)
mae = mean_absolute_error(y_test_reverted, y_pred_reverted)
print(f"El emae es: {mae:.2f}")

El emae es: 20812.25


### 7. Ejercicios Propuestos

<div class="alert alert-block alert-warning">
<h4>Ejercicios para Solidificar Conceptos</h4>
<ol>
    <li><strong>Evaluación de Residuos:</strong> Para el mejor modelo que encontraste (probablemente RidgeCV o LassoCV), calculá los residuos ($y_{test} - \hat{y}$) y creá un gráfico de dispersión de los valores predichos vs. los residuos. ¿Observás algún patrón? Un patrón podría indicar que la relación no es puramente lineal.</li>
    <li><strong>Cambiar la Métrica de CV:</strong> En `LassoCV` y `RidgeCV`, el scoring por defecto es el error cuadrático negativo. Investigá cómo podrías usar el "mean_absolute_error" en su lugar. ¿Cambia el `alpha` seleccionado?</li>
    <li><strong>Implementar K-Fold a Mano:</strong> En lugar de usar `RidgeCV`, instancia un `KFold(n_splits=5)`, iterá sobre los pliegues, entrená un modelo `Ridge` para un alpha fijo en cada pliegue y calculá el error promedio. Compará tu resultado con el de la clase `RidgeCV`.</li>
    <li><strong>Características Polinómicas:</strong> Sospechamos que la relación entre `GrLivArea` y `SalePrice` no es perfectamente lineal. Usá `sklearn.preprocessing.PolynomialFeatures` para crear una característica `GrLivArea` al cuadrado y añadila al modelo. ¿Mejora el R²?</li>
    <li><strong>Impacto de Outliers:</strong> Encontrá la casa más cara en el `df_subset`. Eliminala y volvé a entrenar el modelo OLS (con el target logarítmico). ¿Cómo cambian los coeficientes y el R²?</li>
    <li><strong>Comparar MAE vs MSE:</strong> Calculá el `mean_absolute_error` (MAE) para todos los modelos finales (OLS, RidgeCV, LassoCV) en la escala logarítmica. ¿Por qué el MAE es menor que el MSE?</li>
    <li><strong>Importancia de Características:</strong> Ordená los coeficientes del modelo `RidgeCV` final de mayor a menor en valor absoluto. ¿Cuáles son los 3 features más influyentes según este modelo?</li>
    <li><strong>Efecto del Alpha en Lasso:</strong> Creá un gráfico que muestre cómo cambian los coeficientes de Lasso a medida que `alpha` aumenta. Podés usar la información almacenada en `lasso_cv.path_`.</li>
    <li><strong>Regresión Lineal Simple vs. Múltiple:</strong> Entrená un modelo de regresión lineal simple para cada una de las características en `numeric_features` por separado. Compará el coeficiente de cada característica en su modelo simple con su coeficiente en el modelo múltiple (OLS). ¿Por qué son diferentes?</li>
    <li><strong>Modelo para Producción:</strong> Si tuvieras que elegir uno de los modelos entrenados para ponerlo en producción y predecir precios de casas para un cliente, ¿cuál elegirías y por qué? Justificá tu respuesta basándote en el rendimiento (R², MSE), la interpretabilidad (coeficientes) y la simplicidad del modelo.</li>
</ol>
</div>

In [36]:
# Ejercicio 1
residuos = y_test - y_pred_lasso_cv
px.scatter(x=y_pred_lasso_cv, y=residuos, labels={'x': 'Predicciones', 'y': 'Residuos'})

No observamos ningun patron particular. Si que parece centrado alrededor del 0 sin una forma clara.

In [37]:
# Ejercicio 2
ridge_cv_mae =  RidgeCV(alphas=alphas, store_cv_values=True, scoring='neg_mean_absolute_error')
ridge_cv_mae.fit(X_train_scaled, y_train)

print("\n--- Búsqueda del mejor Alpha para ridge con mae ---")
print(f"Mejor alpha encontrado para Ridge con mae: {ridge_cv_mae.alpha_:.4f}")

# Evaluar el modelo Lasso final
y_pred_ridge_cv_mae = ridge_cv_mae.predict(X_test_scaled)
mse_ridge_cv_mae = mean_squared_error(y_test, y_pred_ridge_cv_mae)
r2_ridge_cv_mae = r2_score(y_test, y_pred_ridge_cv_mae)
print(f"MSE de Lasso con CV: {mse_ridge_cv_mae:.4f}")
print(f"R² de Lasso con CV: {r2_ridge_cv_mae:.4f}")

print(f"Mejor alpha encontrado para Ridge sin mae: {ridge_cv.alpha_:.4f}")


--- Búsqueda del mejor Alpha para ridge con mae ---
Mejor alpha encontrado para Ridge con mae: 37.6494
MSE de Lasso con CV: 0.0299
R² de Lasso con CV: 0.8399
Mejor alpha encontrado para Ridge sin mae: 86.9749


/Users/rodrigoecheconea/Projects/Master AI/aprendizaje_automatico/tp2/.venv/lib/python3.9/site-packages/sklearn/linear_model/_ridge.py:2385: FutureWarning: 'store_cv_values' is deprecated in version 1.5 and will be removed in 1.7. Use 'store_cv_results' instead.
  warnings.warn(


El alpha se redujo considerablemente. Agregar conclusion.
Lasso no soporta cambios en el scoring.

In [38]:
# Ejercicio 3
kf = KFold(n_splits=5)
errors_fold = []
for train_index, test_index in kf.split(X_train_scaled):
  X_train_fold, X_test_fold = X_train_scaled.iloc[train_index], X_train_scaled.iloc[test_index]
  y_train_fold, y_test_fold = y_train.iloc[train_index], y_train.iloc[test_index]

  ridge_model = Ridge(alpha=100)
  ridge_model.fit(X_train_fold, y_train_fold)
  y_pred_ridge_fold = ridge_model.predict(X_test_fold)
  errors_fold.append(mean_squared_error(y_test_fold, y_pred_ridge_fold))

mean_fold_error = np.mean(errors_fold)
print(f"Error promedio de Ridge con KFold: {mean_fold_error:.4f}")

# Comparar con RidgeCV
print(f"Error de RidgeCV: {mse_ridge_cv:.4f}")


  

Error promedio de Ridge con KFold: 0.0311
Error de RidgeCV: 0.0302


Ridge automatico es un 2% mas chico.

In [39]:
# Ejercicio 4
poly = PolynomialFeatures(degree=2, include_bias=False)
grliv_squared = poly.fit_transform(X[['GrLivArea']])[:, 1]

X_squared = X.copy()
X_squared['GrLivArea_squared'] = grliv_squared

X_train_squared, X_test_squared, y_train_squared, y_test_squared = train_test_split(X_squared, y, test_size=0.2, random_state=42)

X_train_squared_scaled = scaler.fit_transform(X_train_squared)
X_test_squared_scaled = scaler.transform(X_test_squared)

ols_model_squared = LinearRegression()
ols_model_squared.fit(X_train_squared_scaled, y_train_squared)
y_pred_ols_squared = ols_model_squared.predict(X_test_squared_scaled)

r2_ols_squared = r2_score(y_test_squared, y_pred_ols_squared)
print(f"R² de OLS con polinomios: {r2_ols_squared:.4f}")
print(f"R² de OLS sin polinomios: {r2_ols:.4f}")












R² de OLS con polinomios: 0.8396
R² de OLS sin polinomios: 0.8410


El resultado es levemente peor por lo cual inferimos que la relacion no es cuadratica.

In [40]:
# Ejercicio 5

df_sin_outlier = df_subset[df_subset['SalePrice'] != df_subset['SalePrice'].max()]
X_sin_outlier_raw = df_sin_outlier[numeric_features]
y_sin_outlier_raw = df_sin_outlier[target_variable]
X_sin_outlier = np.log1p(X_sin_outlier_raw)
y_sin_outlier = np.log1p(y_sin_outlier_raw)

X_train_sin_outlier, X_test_sin_outlier, y_train_sin_outlier, y_test_sin_outlier = train_test_split(X_sin_outlier, y_sin_outlier, test_size=0.2, random_state=42)


X_train_sin_outlier_scaled = scaler.fit_transform(X_train_sin_outlier)
X_test_sin_outlier_scaled = scaler.transform(X_test_sin_outlier)

ols_model_sin_outlier = LinearRegression()
ols_model_sin_outlier.fit(X_train_sin_outlier_scaled, y_train_sin_outlier)
y_pred_ols_sin_outlier = ols_model_sin_outlier.predict(X_test_sin_outlier_scaled)

r2_ols_sin_outlier = r2_score(y_test_sin_outlier, y_pred_ols_sin_outlier)
print(f"R² de OLS sin outliers: {r2_ols_sin_outlier:.4f}")
print(f"R² de OLS con outliers: {r2_ols:.4f}")






R² de OLS sin outliers: 0.8569
R² de OLS con outliers: 0.8410


El R² mejoro, por lo cual era acertada la idea de sacar el outlier

In [43]:
# Ejercicio 6

mae_ols = mean_absolute_error(y_test, y_pred_ols)
mae_lasso_cv = mean_absolute_error(y_test, y_pred_lasso_cv)
mae_ridge_cv = mean_absolute_error(y_test, y_pred_ridge_cv)

print(f"MAE ols: {mae_ols:.4f}")
print(f"MSE ols: {mse_ols:.4f}")
print(f"MAE lasso: {mae_lasso_cv:.4f}")
print(f"MSE lasso: {mse_lasso_cv:.4f}")
print(f"MAE ridge: {mae_ridge_cv:.4f}")
print(f"MSE ridge: {mse_ridge_cv:.4f}")






MAE ols: 0.1200
MSE ols: 0.0297
MAE lasso: 0.1208
MSE lasso: 0.0301
MAE ridge: 0.1204
MSE ridge: 0.0302


Lo que podemos ver es que siempre el MAE es mas grande que el MSE. Esto se debe a que al trabajar con valores menores a 1, elevarlos al cuadrado los achica muchos mas.

In [45]:
# Ejercicio 7

coef_ridge = ridge_cv.coef_
coef_ridge_sorted = pd.DataFrame(coef_ridge, index=numeric_features, columns=['Coeficiente'])
coef_ridge_sorted = coef_ridge_sorted.sort_values(by='Coeficiente', ascending=False)
print(coef_ridge_sorted)






             Coeficiente
OverallQual     0.133732
GrLivArea       0.106413
GarageCars      0.070178
YearBuilt       0.069926
TotalBsmtSF     0.042356
LotArea         0.032643
FullBath        0.007107
YrSold         -0.000008


Aca podemos ver claramente que los mejores coeficientes son (en orden):
- OverallQual
- GrLivArea
- GarageCars

In [50]:
# Ejercicio 8

alphas_path, coefs_path, _ = lasso_path(
    X_train_scaled.values,
    y_train.values,
    alphas=alphas
)

fig = go.Figure()
for i, feature in enumerate(numeric_features):
    fig.add_trace(go.Scatter(
        x=alphas_path,
        y=coefs_path[i, :],
        mode='lines',
        name=feature
    ))

fig.update_xaxes(type='log', title='Alpha')
fig.update_yaxes(title='Coeficiente')
fig.update_layout(title='Coeficientes de Lasso vs Alpha')
fig.show()


Al ir aumentando los valores de alfa, los coeficientes de las diversas caracteristicas caen a 0, cayendo antes los menos importantes (se puede ver que fullBath es el primero en caer, tal como analizamos antes).

In [ ]:
# Ejercicio 9

resultados = []
for feature in numeric_features:
    X_train_modelo_simple = X_train_scaled[[feature]]
    X_test_modelo_simple = X_test_scaled[[feature]]

    modelo = LinearRegression()
    modelo.fit(X_train_modelo_simple, y_train)
    predicciones = modelo.predict(X_test_modelo_simple)

    coeficiente = modelo.coef_[0]
    coeficiente_multiple = ols_model.coef_[numeric_features.index(feature)]

    resultados.append({
        'Caracteristica': feature,
        'Coeficiente_Simple': coeficiente,
        'Coeficiente_Multiple': coeficiente_multiple
    })

df_resultados = pd.DataFrame(resultados)
print(df_resultados)





    

IndexError: only integers, slices (`:`), ellipsis (`...`), numpy.newaxis (`None`) and integer or boolean arrays are valid indices